In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install llama-index
!pip install llama-index-llms-groq
!pip install llama-index-embeddings-huggingface
!pip install llama-index-retrievers-bm25
!pip install llama-index-postprocessor-cohere-rerank
!pip install faiss-cpu
!pip install pypdf
!pip install gradio

In [ ]:
import os

# 🔥 Groq API
os.environ["GROQ_API_KEY"] = "xxxxx"

# 🔥 Cohere API
COHERE_API_KEY = "XXXXXX"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/MyDrive/Dataset.pdf"
STORAGE_PATH = "/content/drive/MyDrive/MyDrive"

In [ ]:
!pip install llama-index llama-index-embeddings-huggingface llama-index-llms-groq llama-index-vector-stores-faiss

In [ ]:
from pypdf import PdfReader

PDF_PATH = "/content/drive/MyDrive/MyDrive/Dataset.pdf"   # 👈 ADD THIS

reader = PdfReader(PDF_PATH)

print("Total pages:", len(reader.pages))

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

In [ ]:
import gradio as gr

def chat_fn(message, history):
    response = query_engine.query(message)
    return str(response)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="📚 UPSC Advanced RAG Assistant",
    description="Hybrid Search + Reranker + Groq"
)

demo.launch(share=True)

In [ ]:
import faiss
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore
from llama_index.core.node_parser import SimpleNodeParser

# 🔥 Embedding model (fast + good)
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# 🔥 Chunking
parser = SimpleNodeParser.from_defaults(
    chunk_size=600,
    chunk_overlap=60
)

# 🔥 FAISS (FIXED)
faiss_index = faiss.IndexFlatL2(384)
vector_store = FaissVectorStore(faiss_index=faiss_index)

storage_context = StorageContext.from_defaults(
    vector_store=vector_store
)

# 🔥 Incremental indexing (NO CRASH)
BATCH_SIZE = 10
index = None

for i in range(0, len(reader.pages), BATCH_SIZE):
    print(f"Processing pages {i} to {i+BATCH_SIZE}")

    batch_docs = []

    for j in range(i, min(i+BATCH_SIZE, len(reader.pages))):
        text = reader.pages[j].extract_text()

        if text:
            batch_docs.append(Document(text=text))

    if not batch_docs:
        continue

    nodes = parser.get_nodes_from_documents(batch_docs)

    if index is None:
        index = VectorStoreIndex(
            nodes,
            storage_context=storage_context,
            embed_model=embed_model
        )
    else:
        index.insert_nodes(nodes)

# 💾 Save index
index.storage_context.persist(persist_dir=STORAGE_PATH)

print("✅ Index created successfully!")

In [ ]:
from llama_index.llms.groq import Groq
from llama_index.core import Settings

llm = Groq(model="llama-3.1-8b-instant")   # 🔥 recommended

Settings.llm = llm

In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever

# Vector retriever
vector_retriever = index.as_retriever(similarity_top_k=3)

# BM25 retriever
bm25_retriever = BM25Retriever.from_defaults(
    docstore=index.docstore,
    similarity_top_k=3
)

# 🔥 Hybrid retriever
retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=5,   # 🔥 BEST VALUE
    mode="reciprocal_rerank"
)

In [ ]:
from llama_index.postprocessor.cohere_rerank import CohereRerank

COHERE_API_KEY = "xxxxxx"

reranker = CohereRerank(
    top_n=3,
    api_key=COHERE_API_KEY
)

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

In [ ]:
query_engine = RetrieverQueryEngine.from_args(
    retriever=retriever,
    node_postprocessors=[reranker]
)

In [ ]:
response = query_engine.query("Explain fundamental rights in India")

print(response)

In [ ]:
import gradio as gr

def chat_fn(message, history):
    response = query_engine.query(message)
    return str(response)

demo = gr.ChatInterface(
    fn=chat_fn,
    title="📚 UPSC Advanced RAG Assistant",
    description="Hybrid Search + Cohere Reranker + Groq"
)

demo.launch(share=True)